# 6.4. Lazy Initialization
D2L의 Lazy Initialization장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Lazy Initialization

지금까지 `nn.Linear()`를 사용할 때는 보통 입력 feature 개수를 직접 지정했다.

    nn.Linear(20, 256)

입력 feature : 20개, 출력 뉴런 : 256개
weight.shape : [256, 20], bias.shape : [256]

그런데 가끔 모델을 만들 때 입력 feature가 몇 개인지 아직 모르는 경우가 있다. 이때 PyTorch `LazyLinear`를 사용할 수 있다.

    nn.LazyLinear

입력 feature : 아직 모름, 출력 뉴런 : 256개

실제 입력 데이터가 들어오면 PyTorch가 입력 크기를 보고 자동으로 결정한다.

## 2. 왜 Lazy Initialization이 필요할까?

일반적인 Linear layer는 이런식으로 쓴다.

    nn.Linear(20, 256)

그러려면 모델을 만들 때부터 입력이

    [batch_size, 20]

형태인 것을 알고 있어야 한다. 하지만 복잡한 모델에서는 이전 layer가 정확히 몇 개의 feature를 출력하는지 직접 계산하기 번거로운 경우가 있다.

`Lazy Initialization`을 사용하면 출력 크기만 지정해놓고, 첫 forward에서 입력 크기를 자동으로 알아낼 수 있다.

D2L에서는 특히 CNN처럼 이전 연산 결과의 차원을 직접 계산하기 번거로운 구조에서 이런 방식이 모델 수정과 설계를 편하게 할 수 있다고 한다.

## 3. LazyLinear로 MLP 만들기

In [2]:
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

net

Sequential(
  (0): LazyLinear(in_features=0, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)

그러면 모델 구조는 이렇다.

```text
입력
 ↓
LazyLinear(? → 256)
 ↓
ReLU
 ↓
LazyLinear(? → 10)
 ↓
출력
```

두개 layer의 입력 feature 수를 지정하지 않았다.

## 4. 아직 Weight가 만들어지지 않은 상태

아직 데이터를 넣지 않아서 PyTorch는 첫 번째 layer가 몇 개의 입력 feature를 받아야 하는지 모른다.

In [3]:
print(net[0].weight)

<UninitializedParameter>


## 5. 데이터 넣어보기

입력 데이터를 만들어보자

In [ ]:
X = torch.rand(2, 20)

print(X.shape) # batch_size = 2, feature = 20

torch.Size([2, 20])


In [ ]:
y = net(X) # 모델에 넣기

모델에 넣는 순간 PyTorch는 입력 feature가 20개인걸 알 수 있다. 따라서 첫 번째 LazyLinear를 실제 Linear layer 형태로 결정할 수 있다.

## 6. Weight Shape는 어떻게 결정되나?

첫 forward가 끝난 후 확인해보면

In [6]:
print(net[0].weight.shape)
print(net[0].bias.shape)

print(net[2].weight.shape)
print(net[2].bias.shape)

torch.Size([256, 20])
torch.Size([256])
torch.Size([10, 256])
torch.Size([10])


첫 번째 Layer

입력 X = [2, 20]

nn.LazyLinear(256) 이 (20, 256)으로 결정되서
weight.shape = [256, 20], bias.shape = [256]

두 번째 Layer

첫 번째 layer가 출력 하는 값 = [2, 256]

nn.LazyLinear(10) 이 (256, 10)으로 결정된다.
weight.shape = [10, 256], bias.shape = [10]

첫 입력 shape를 알아낸 뒤, 그 정보가 앞 layer부터 뒤 layer까지 순서대로 전달되면서 모든 파라미터 shape이 결정된다.

## 7. 전체 Shape 흐름

In [7]:
X = torch.rand(2, 20)

y = net(X)

print("입력:", X.shape)
print("첫 번째 weight:", net[0].weight.shape)
print("두 번째 weight:", net[2].weight.shape)
print("출력:", y.shape)

입력: torch.Size([2, 20])
첫 번째 weight: torch.Size([256, 20])
두 번째 weight: torch.Size([10, 256])
출력: torch.Size([2, 10])


```text
X
[2, 20]

    ↓ LazyLinear(256)

[2, 256]

    ↓ ReLU

[2, 256]

    ↓ LazyLinear(10)

[2, 10]
```

첫 batch_size 2는 weight의 크기를 결정하지 않는다. Linear layer에서 중요한 것은 feature 차원이다.

    [batch_size, features(중요)]

## 8. Linear와 LazyLinear 비교

두 코드를 비교하면 엄청 간단하다.

```py
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
) # 개발자가 직접 20 -> 256 -> 10 모두 지정
```

```py
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
) # 개발자는 ? -> 256 -> 10 처럼 작성할 수 있다.

X = torch.rand(2, 20) # 실행하는 순간 20 -> 256 -> 10 으로 결정된다.
net(X) # 자동 shape 추론
```

## 9. Dummy Input으로 먼저 초기화하기

실제 학습 전에 가짜 데이터를 한 번 넣어서 모델 shape를 미리 결정할 수도 있다.

이런 작업을 모델의 dry run처럼 생각하면 된다. D2L에서는 dummy input을 한 번 forward시켜 모든 parameter shape를 추론한 뒤 초기화 하는 방식을 설명한다.

In [ ]:
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

dummy_X = torch.zeros(1, 20) # 임시 데이터를 넣었지만 
# feature가 20이라는 사실은 알수 있기 때문에 parameter shape 결정 가능

net(dummy_X)

print(net[0].weight.shape)
print(net[2].weight.shape)

torch.Size([256, 20])
torch.Size([10, 256])


## 10. 오늘의 정리

- Lazy Initialization은 파라미터 초기화를 첫 입력이 들어올 때까지 미루는 방식이다.
- nn.Linear(20, 256)은 입력 feature를 직접 지정한다.
- nn.LazyLinear(256)은 입력 feature를 지정하지 않는다.
- LazyLinear는 첫 forward에서 입력 tensor의 feature 크기를 확인한다.
- 입력이 [2, 20]이면 LazyLinear(256)의 weight는 [256, 20]이 된다.
- 다음 layer의 입력 크기는 이전 layer의 출력 크기로 자동 결정된다.
- 따라서 ? → 256 → 10으로 만든 모델도 첫 forward 후 20 → 256 → 10으로 확정될 수 있다.
- Lazy Initialization의 핵심 장점은 layer의 입력 차원을 직접 계산해야 하는 부담을 줄이는 것이다.